# Multiple Categories

This chapter examines how to compare categorical distributions across multiple groups. We will use jury-selection data from Alameda County and assess whether observed jury panels are consistent with random selection from the eligible population.

In [ ]:
import pandas as pd
%matplotlib inline
path_data = "../../../../data/"
import matplotlib.pyplot as plt
plt.style.use("fivethirtyeight")
import numpy as np

## Jury Selection in Alameda County

The ACLU of Northern California studied the racial and ethnic composition of jury panels in Alameda County. The data compare the estimated distribution of eligible jurors with the distribution of people who appeared for jury service.

In [ ]:
jury = pd.DataFrame({
    'Ethnicity': ['Asian/PI','Black/AA','Caucasian','Hispanic','Other'],
    'Eligible': [0.15,0.18,0.54,0.12,0.01],
    'Panels': [0.26,0.08,0.54,0.08,0.04]
})
jury

Some groups are overrepresented and some are underrepresented in the panel data.

In [ ]:
jury.set_index('Ethnicity')[['Eligible','Panels']].plot.barh()
plt.show()

## Comparison with Panels Selected at Random

We will compare the panel distribution to what we would expect from a random sample of 1,453 people drawn from the eligible population.

In [ ]:
def sample_proportions(sample_size, proportions):
    categories = np.arange(len(proportions))
    sample = np.random.choice(categories, size=sample_size, p=proportions)
    counts = np.bincount(sample, minlength=len(proportions))
    return counts / sample_size

In [ ]:
eligible_population = jury['Eligible'].to_numpy()
sample_distribution = sample_proportions(1453, eligible_population)
panels_and_sample = jury.copy()
panels_and_sample['Random Sample'] = sample_distribution
panels_and_sample

In [ ]:
panels_and_sample.set_index('Ethnicity')[['Eligible','Panels','Random Sample']].plot.barh()
plt.show()

The random sample resembles the eligible population much more closely than the observed panels.

## A New Statistic: Total Variation Distance

To compare distributions quantitatively, we use the **total variation distance (TVD)**.

For two distributions, TVD is half the sum of the absolute differences between corresponding category proportions.

In [ ]:
jury_with_diffs = jury.copy()
jury_with_diffs['Difference'] = jury['Panels'] - jury['Eligible']
jury_with_diffs

In [ ]:
jury_with_diffs['Absolute Difference'] = np.abs(jury_with_diffs['Difference'])
jury_with_diffs

In [ ]:
jury_with_diffs['Absolute Difference'].sum()/2

The observed TVD is 0.14.

In [ ]:
def total_variation_distance(distribution_1, distribution_2):
    return np.sum(np.abs(distribution_1 - distribution_2)) / 2

In [ ]:
total_variation_distance(jury['Panels'].to_numpy(), jury['Eligible'].to_numpy())

In [ ]:
sample_distribution = sample_proportions(1453, eligible_population)
total_variation_distance(sample_distribution, eligible_population)

## Simulating the Statistic Under the Model

We now simulate the TVD repeatedly under the assumption that panelists are selected at random from the eligible population.

In [ ]:
def one_simulated_tvd():
    sample_distribution = sample_proportions(1453, eligible_population)
    return total_variation_distance(sample_distribution, eligible_population)

In [ ]:
tvds = np.array([])
repetitions = 5000
for i in np.arange(repetitions):
    tvds = np.append(tvds, one_simulated_tvd())

## Assessing the Model of Random Selection

In [ ]:
pd.DataFrame({'TVD': tvds})['TVD'].plot.hist(bins=np.arange(0,0.2,0.005))
plt.title('Prediction Assuming Random Selection')
plt.xlim(0,0.15)
plt.scatter(0.14,0,color='red',s=40)
plt.show()

The observed TVD of 0.14 lies far beyond the bulk of the simulated values. This provides evidence that the observed panel composition is not consistent with the model of random selection from the eligible population.

## Reasons for the Bias

Statistical analyses can identify discrepancies between observed data and a model, but they do not by themselves determine the causes. The ACLU report discusses potential contributors including panel-selection procedures, source lists used to identify potential jurors, response rates to jury summons, economic barriers to participation, and historical inequities that affect representation.

## Data Quality

As in any data-science investigation, conclusions depend on the quality of the underlying data. The estimated eligible-juror distribution is itself an estimate, and the reported panel demographics depend on classification methods and response patterns. These considerations should be kept in mind when interpreting results.

## Conclusion

The simulation shows that the reported jury-panel distribution does not resemble what would typically arise from random sampling of the estimated eligible-juror population. The observed discrepancy is much larger than would be expected due to chance alone under the model.